# Project 04 — House Price Predictor
**Skills demonstrated:** EDA · Feature Engineering · Linear Regression · Model Evaluation  
**Dataset:** California Housing (built into scikit-learn — no download needed)  
**Goal:** Predict median house prices from neighbourhood features, evaluate the model honestly, and communicate findings clearly.

---
## The Engineering Mindset Going In
Before touching code, ask: *What exactly are we predicting, and why does it matter?*

- **Target (y):** Median house value for a block (in $100,000s)
- **Features (X):** 8 numerical inputs — income, house age, rooms, bedrooms, population, occupancy, latitude, longitude
- **Problem type:** Regression — output is a continuous number, not a category
- **Success metric:** RMSE (Root Mean Squared Error) + R² — both reported, both explained

---
## Step 1 — Import Libraries
We import only what we need. Every library has a job.

In [ ]:
# ── Data handling ──────────────────────────────────────────────────────────────
import pandas as pd          # DataFrames — our main tool for working with tabular data
import numpy as np           # Numerical operations — used for calculations

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt   # Base plotting library
import seaborn as sns             # Statistical visualisation built on matplotlib

# ── Dataset ────────────────────────────────────────────────────────────────────
from sklearn.datasets import fetch_california_housing  # Built-in real dataset

# ── ML pipeline ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split   # Split data into train/test
from sklearn.preprocessing import StandardScaler       # Scale features to same range
from sklearn.linear_model import LinearRegression      # Our model
from sklearn.metrics import mean_squared_error, r2_score  # Evaluation metrics

# ── Settings ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')   # Keep output clean

# Set a consistent visual style for all charts
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('✓ All libraries imported successfully')

---
## Step 2 — Load & Inspect the Data
Before any analysis, always answer: *What shape is the data? Are there missing values? What does each column mean?*

This is called a **Data Audit** — you did this in Project 01. Same discipline here.

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
# fetch_california_housing() returns a Bunch object (like a dictionary)
# as_frame=True converts it directly to a pandas DataFrame — easier to work with
housing = fetch_california_housing(as_frame=True)

# Combine features (X) and target (y) into one DataFrame for EDA
df = housing.frame.copy()

# ── Basic inspection ───────────────────────────────────────────────────────────
print('━' * 50)
print(f'Rows:     {df.shape[0]:,}')   # How many data points?
print(f'Columns:  {df.shape[1]}')     # How many features + target?
print('━' * 50)
print('\nColumn names and data types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())   # Critical check — missing values break models
print('━' * 50)

In [ ]:
# ── What does each column mean? ────────────────────────────────────────────────
# Always document this — a hiring manager reading your notebook needs to understand
# the domain before they can evaluate your analysis

column_descriptions = {
    'MedInc':       'Median income in block group (in tens of thousands $)',
    'HouseAge':     'Median house age in block group (years)',
    'AveRooms':     'Average number of rooms per household',
    'AveBedrms':    'Average number of bedrooms per household',
    'Population':   'Block group population',
    'AveOccup':     'Average number of household members',
    'Latitude':     'Block group latitude (geographic position)',
    'Longitude':    'Block group longitude (geographic position)',
    'MedHouseVal':  'TARGET — Median house value (in $100,000s)'
}

print('Data Dictionary:\n')
for col, desc in column_descriptions.items():
    print(f'  {col:<14} → {desc}')

# ── Preview first 5 rows ───────────────────────────────────────────────────────
print('\nFirst 5 rows:')
df.head()

---
## Step 3 — Exploratory Data Analysis (EDA)
EDA answers: *Where is the data strange? What patterns exist? What should I investigate before modelling?*

We produce 4 charts — each one earns its place by telling us something actionable.

In [ ]:
# ── Chart 1 — Distribution of the TARGET variable ─────────────────────────────
# ALWAYS look at your target first. Is it normally distributed? Skewed? Capped?
# A skewed target can mislead regression metrics.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution of Median House Value (Target)', fontweight='bold')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Count')

# Box plot — shows outliers clearly
axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Box Plot — Outlier Detection', fontweight='bold')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.savefig('chart_01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Observation ───────────────────────────────────────────────────────────────
# Notice the spike at 5.0 — this is a data cap (values above $500,000 were recorded
# as 5.0). This is important to note as a limitation in your README.
print(f'\nTarget statistics:')
print(df['MedHouseVal'].describe().round(3))
print(f'\nOBSERVATION: Values capped at 5.0 ({(df["MedHouseVal"] == 5.0).sum()} rows)')
print('This is a known data quality issue — document it in the README.')

In [ ]:
# ── Chart 2 — Correlation Heatmap ─────────────────────────────────────────────
# Which features have the strongest LINEAR relationship with house value?
# Correlation ranges from -1 (inverse) to +1 (direct). Near 0 = weak relationship.

plt.figure(figsize=(10, 7))

# Calculate correlation matrix
corr_matrix = df.corr()

# Draw heatmap — annot=True shows the numbers inside each cell
sns.heatmap(
    corr_matrix,
    annot=True,           # Show correlation values in each cell
    fmt='.2f',            # Round to 2 decimal places
    cmap='coolwarm',      # Blue = negative, Red = positive correlation
    center=0,             # White = zero correlation
    linewidths=0.5,
    square=True
)

plt.title('Feature Correlation Matrix\n(Focus on MedHouseVal row)', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Extract and print correlations with target only ───────────────────────────
target_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
print('\nCorrelation with MedHouseVal (strongest to weakest):')
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 20)
    direction = '+' if val > 0 else '-'
    print(f'  {feat:<14} {direction}{bar:<20} {val:.3f}')

In [ ]:
# ── Chart 3 — Income vs House Value (strongest predictor) ─────────────────────
# MedInc has the highest correlation. Let's visualise that relationship directly.
# This is the kind of chart that makes sense to a non-technical stakeholder.

plt.figure(figsize=(10, 5))

# Sample 2000 points for speed — plotting all 20,000 makes it slow
sample = df.sample(2000, random_state=42)

plt.scatter(
    sample['MedInc'],
    sample['MedHouseVal'],
    alpha=0.3,           # Transparency — shows density
    color='steelblue',
    edgecolors='none',
    s=15                 # Dot size
)

plt.xlabel('Median Income (tens of thousands $)', fontsize=11)
plt.ylabel('Median House Value ($100,000s)', fontsize=11)
plt.title('Income vs House Value — Clear Positive Relationship\n(capping at 5.0 visible as horizontal band)', fontweight='bold')

# Add a horizontal line at the cap to highlight the data quality issue
plt.axhline(y=5.0, color='red', linestyle='--', alpha=0.5, label='Data cap at $500K')
plt.legend()

plt.tight_layout()
plt.savefig('chart_03_income_vs_value.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 4 — Geographic Distribution of House Values ─────────────────────────
# Latitude and Longitude are features — but together they show geography.
# This is feature engineering thinking: two columns combined reveal something
# neither shows alone.

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    df['Longitude'],
    df['Latitude'],
    c=df['MedHouseVal'],   # Colour = house value
    cmap='plasma',          # Yellow = high value, purple = low value
    alpha=0.4,
    s=5
)

plt.colorbar(scatter, label='Median House Value ($100,000s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('California House Values by Geographic Location\n'
          'Yellow = High Value (Coastal) · Purple = Lower Value (Inland)',
          fontweight='bold')

plt.tight_layout()
plt.savefig('chart_04_geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('OBSERVATION: Coastal areas (San Francisco, Los Angeles) show clearly higher values.')
print('This confirms that Latitude and Longitude carry real predictive signal.')

---
## Step 4 — Feature Engineering & Cleaning
Based on EDA findings, we make deliberate decisions before training.

**Decisions made here:**
1. Remove the capped rows (MedHouseVal == 5.0) — they are not real values, they are a data quality artefact
2. Cap extreme outliers in AveRooms and AveOccup — these skew the model
3. Scale all features — Linear Regression is sensitive to feature magnitude

In [ ]:
# ── Decision 1 — Remove capped target values ──────────────────────────────────
# Rows where MedHouseVal == 5.0 are artificially capped, not real prices.
# Training on these would teach the model a false pattern.

rows_before = len(df)
df_clean = df[df['MedHouseVal'] < 5.0].copy()   # Keep only uncapped rows
rows_removed = rows_before - len(df_clean)

print(f'Rows before cleaning:  {rows_before:,}')
print(f'Rows removed (capped): {rows_removed:,}')
print(f'Rows after cleaning:   {len(df_clean):,}')

# ── Decision 2 — Cap extreme outliers in AveRooms and AveOccup ────────────────
# Values like AveRooms=100 are data errors (an average block cannot have 100 rooms/house)
# We cap at the 99th percentile — preserves real variation, removes errors

for col in ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']:
    cap = df_clean[col].quantile(0.99)   # 99th percentile value
    before_cap = (df_clean[col] > cap).sum()
    df_clean[col] = df_clean[col].clip(upper=cap)
    print(f'  {col:<14}: capped {before_cap} extreme values at {cap:.1f}')

print(f'\n✓ Cleaning complete. Working dataset: {len(df_clean):,} rows')

In [ ]:
# ── Separate features (X) and target (y) ──────────────────────────────────────
# This is the standard ML split — always done before train/test split

X = df_clean.drop('MedHouseVal', axis=1)   # All columns EXCEPT the target
y = df_clean['MedHouseVal']                # Only the target column

print(f'Features (X) shape: {X.shape}   ← rows × columns')
print(f'Target   (y) shape: {y.shape}   ← one value per row')
print(f'\nFeatures used: {list(X.columns)}')

# ── Train / Test split ────────────────────────────────────────────────────────
# 80% for training, 20% for testing
# random_state=42 — ensures you get the same split every time you run this
# This is critical for reproducibility — a hiring manager should get the same results

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% held out for testing
    random_state=42       # Fixed seed — reproducible results
)

print(f'\nTraining set:  {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:      {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X)*100:.0f}%)')
print('\nIMPORTANT: The test set is sealed until Step 6. We do NOT look at it before then.')

In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────────
# Linear Regression is sensitive to feature scale.
# MedInc ranges 0-15 but Population ranges 0-35,000 — without scaling,
# Population dominates just because of its magnitude, not its importance.
#
# StandardScaler: transforms each feature to mean=0, std=1
# CRITICAL RULE: fit_transform on TRAIN only. transform (not refit) on TEST.
# Fitting on test = data leakage (your model sees future information)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Learn scale from train, apply it
X_test_scaled  = scaler.transform(X_test)         # Apply the SAME scale — never refit

# Verify scaling worked
print('Feature means after scaling (should be ~0.0):')
means = X_train_scaled.mean(axis=0).round(3)
for name, mean in zip(X.columns, means):
    print(f'  {name:<14}: {mean}')

print('\n✓ Scaling complete — all features now on the same scale')

---
## Step 5 — Train the Model
Linear Regression fits a straight line (or hyperplane in multiple dimensions) through the data.
The model learns the **weight** (coefficient) for each feature — how much house value changes per unit of that feature.

In [ ]:
# ── Train Linear Regression ───────────────────────────────────────────────────
# model.fit() is where learning happens:
# The algorithm finds weights (w) and bias (b) that minimise MSE across all training rows
# Formula: y = w1*MedInc + w2*HouseAge + ... + w8*Longitude + b

model = LinearRegression()
model.fit(X_train_scaled, y_train)   # PARAMETERS (weights) are learned here

print('✓ Model trained')
print(f'\nModel learned {len(model.coef_)} weights (one per feature) + 1 bias term')
print(f'Bias (intercept): {model.intercept_:.4f}')

# ── Interpret the coefficients ────────────────────────────────────────────────
# Coefficient = how much house value changes for a 1 standard-deviation increase in that feature
# Positive = higher feature → higher house value
# Negative = higher feature → lower house value

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

print('\nFeature Coefficients (effect on house value per 1 std dev increase):')
for _, row in coef_df.iterrows():
    direction = '▲' if row['Coefficient'] > 0 else '▼'
    bar = '█' * int(abs(row['Coefficient']) * 5)
    print(f'  {row["Feature"]:<14} {direction} {bar:<15} {row["Coefficient"]:+.4f}')

---
## Step 6 — Evaluate the Model
Three metrics + a residual plot. Each one tells us something different.

| Metric | What it tells you | Good value |
|--------|------------------|------------|
| **RMSE** | Average error in original units ($100K) | As low as possible |
| **MAE** | Average absolute error | As low as possible |
| **R²** | % of variance explained by the model | Closer to 1.0 is better |

In [ ]:
# ── Generate predictions on the test set ──────────────────────────────────────
# This is the first time we use the test set — the model has never seen this data

y_pred = model.predict(X_test_scaled)   # Model predicts house values for test rows

# ── Calculate metrics ─────────────────────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(y_test, y_pred))   # Root Mean Squared Error
mae  = np.mean(np.abs(y_test - y_pred))              # Mean Absolute Error
r2   = r2_score(y_test, y_pred)                      # R-squared

print('=' * 50)
print('MODEL EVALUATION — TEST SET RESULTS')
print('=' * 50)
print(f'  RMSE : {rmse:.4f}  (${rmse*100_000:,.0f} average error)')
print(f'  MAE  : {mae:.4f}  (${mae*100_000:,.0f} average absolute error)')
print(f'  R²   : {r2:.4f}  ({r2*100:.1f}% of variance explained)')
print('=' * 50)
print(f'\nINTERPRETATION:')
print(f'  On average, this model\'s predictions are off by ${rmse*100_000:,.0f}.')
print(f'  The model explains {r2*100:.1f}% of the variation in house prices.')
print(f'  The remaining {(1-r2)*100:.1f}% is driven by factors not in this dataset')
print(f'  (e.g. school quality, proximity to amenities, property condition).')

In [ ]:
# ── Residual Plot — the most important diagnostic chart ───────────────────────
# Residuals = actual - predicted
# A good model has residuals randomly scattered around zero (Homoscedasticity)
# A pattern in residuals = the model is missing something systematic

residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart A — Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.3, s=8, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Values ($100,000s)')
axes[0].set_ylabel('Residuals (Actual − Predicted)')
axes[0].set_title('Residual Plot\nRandom cloud = good · Pattern = model is missing something',
                  fontweight='bold')

# Chart B — Actual vs Predicted (how close to the perfect line?)
axes[1].scatter(y_test, y_pred, alpha=0.3, s=8, color='darkorange')

# Perfect prediction line — if model was perfect, all dots would lie on this
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual Values ($100,000s)')
axes[1].set_ylabel('Predicted Values ($100,000s)')
axes[1].set_title(f'Actual vs Predicted\nR² = {r2:.3f}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_05_residuals_and_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print('WHAT TO LOOK FOR:')
print('  Left chart:  If dots form a fan or curve → model has a systematic failure')
print('  Right chart: Dots close to the red line → good predictions')
print('  Notice the horizontal band at y=5.0 — residuals pile up there')
print('  This is the data cap we identified in EDA. Our model cannot predict beyond $500K.')

---
## Step 7 — Findings & Limitations
This section is what separates a junior from a professional. Anyone can run sklearn. 
**The skill is knowing what your results mean and what they do not mean.**

In [ ]:
# ── Summary of findings ───────────────────────────────────────────────────────
# Write this as if explaining to a non-technical stakeholder.
# This is what goes in your README.

findings = {
    'Finding 1': f'Median income is the strongest predictor of house value (corr: {df.corr()["MedHouseVal"]["MedInc"]:.2f}). '
                  'For every 1 standard deviation increase in neighbourhood income, '
                 f'house value increases by ${model.coef_[0]*100_000:,.0f} on average.',

    'Finding 2': 'Geographic location (Latitude/Longitude) carries significant predictive power. '
                 'Coastal California blocks command substantially higher prices, '
                 'which aligns with known real estate patterns in the region.',

    'Finding 3': f'The model explains {r2*100:.1f}% of house price variation (R² = {r2:.3f}). '
                  'The remaining variance is driven by factors not in this dataset: '
                  'school district quality, walkability, property condition, and local amenities.',

    'Finding 4': f'Average prediction error is ${rmse*100_000:,.0f} (RMSE). '
                  'For a mid-range property at $250,000, this represents a '
                 f'{rmse*100_000/250_000*100:.0f}% error margin — acceptable for macro analysis, '
                  'insufficient for individual property valuation.',

    'Limitation 1': 'The dataset caps all values above $500,000 at exactly $500,000. '
                     'This model cannot predict high-end property values reliably. '
                     'A production model would require uncapped data.',

    'Limitation 2': 'This is a 1990 census dataset. Applying it to current markets '
                     'would produce unreliable results — this is a classic example of '
                     'model drift: the world has changed but the model has not.'
}

print('PROJECT FINDINGS & LIMITATIONS')
print('=' * 60)
for title, text in findings.items():
    print(f'\n{title}:')
    # Wrap text for readability
    words = text.split()
    line = '  '
    for word in words:
        if len(line) + len(word) > 70:
            print(line)
            line = '  ' + word + ' '
        else:
            line += word + ' '
    print(line)

---
## ✓ Project Complete

**What this notebook demonstrates:**
- Data loading and inspection (audit mindset)
- EDA with 4 charts — each one earning its place
- Feature engineering and deliberate cleaning decisions
- Correct train/test split with no data leakage
- StandardScaler applied correctly (fit on train only)
- Linear Regression — trained, coefficients interpreted
- RMSE, MAE, R² — calculated and explained in plain English
- Residual analysis — model diagnostic, not just a metric
- Honest limitations — what the model cannot do

**Next step:** Copy the findings from Step 7 into your README and push to GitHub.